# Ablation Study — AugCRNN-T (TAM VERİ)

> **Önemli:** Önceki tüm sayılar (78.06 / 84.54, N=5,338) IAM'in **kesik** bir
> etiket dosyasıyla elde edilmişti. Repo artık tam IAM etiketlerinden kurulmuş
> split dosyalarını taşıyor (`aachen_splits/*_words.txt`). Bu yüzden **her şey
> sıfırdan, tam veriyle** ölçülür. Kaggle'daki `words.txt` artık kullanılmıyor;
> sadece görüntüler kullanılıyor.

| Split | Form | Kelime (ok) |
|---|---:|---:|
| train | 747 | 47,999 |
| validation | 111 | 7,205 |
| test | 336 | **20,310** |

Resmi Aachen listesi 747/116/336 formdur; validation'dan **5 form** test ile metin
(prompt) örtüşmesi nedeniyle çıkarılmıştır (116 → 111). Test setine dokunulmamıştır.

Tam veri ~1.5× büyük, bir eğitim ~3–3.5 saat. Kaggle oturumu 12 saat olduğu için
**iki oturum** gerekiyor. `SESSION` değişkeniyle seçilir.

## Oturum A — ana sayılar (2 eğitim, ~7 saat)
| `--aug-mode` | Anlamı |
|---|---|
| `narrow` | **CRNN-L baseline** (dar fotometrik, elastik ✗, morf ✗) |
| `full` | **AugCRNN-T** (önerilen) |

`full` eğitimi biterken `narrow` çıktısına karşı **McNemar** testi otomatik koşar.
Ardından Tablo B (lexicon ablation) `full` modelinin üstünde çalışır (~20 dk).

## Oturum B — bileşen ablation'ı (3 eğitim, ~10 saat)
| `--aug-mode` | Anlamı |
|---|---|
| `photo` | geniş fotometrik, elastik ✗, morf ✗ |
| `elastic` | geniş fotometrik + elastik |
| `morph` | geniş fotometrik + morfolojik |

## Gerekli Input
- IAM word dataset (`words/` görüntü klasörü olan herhangi biri; `words.txt`'si kesik olsa da fark etmez)

## Settings
Accelerator **GPU T4**, Internet **ON**, **Save & Run All (Commit)**

In [ ]:
# Hücre 1: oturum seçimi + ortam + repo
SESSION = "A"          # "A" = narrow + full + lexicon ablation   |   "B" = photo + elastic + morph

import torch, sys, os, subprocess, shutil, json, time
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}  |  PyTorch {torch.__version__}  |  SESSION {SESSION}")

REPO_URL = "https://github.com/Ridvan013/CRNN-Handwriting-Recognition.git"
BRANCH   = "feature/aachen-v3-extended-trigram"
REPO_DIR = "/kaggle/working/repo"
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,REPO_DIR], check=True)
SHA = subprocess.run(["git","-C",REPO_DIR,"rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print(f"repo commit: {SHA}")
for name in ["cloud","aachen_splits","trigram_lm.py","verify_aachen_splits.py"]:
    src, dst = os.path.join(REPO_DIR,name), os.path.join("/kaggle/working",name)
    shutil.copytree(src,dst,dirs_exist_ok=True) if os.path.isdir(src) else shutil.copy(src,dst)
sys.path.insert(0,"/kaggle/working"); os.chdir("/kaggle/working")
# klon artık gerekmiyor; .git + eski .pth dosyalari commit ciktisini sisirmesin
shutil.rmtree(REPO_DIR)

# split dosyalari TAM veriden mi? (kesik etiket dosyasiyla test 5,338 olurdu)
EXPECTED = {"train_words.txt": 47999, "validation_words.txt": 7205, "test_words.txt": 20310}
for fn, exp in EXPECTED.items():
    n = sum(1 for l in open(f"aachen_splits/{fn}") if l.strip() and not l.startswith("#"))
    assert n == exp, f"{fn}: {n} satir — repo eski/bozuk! ({exp:,} bekleniyor)"
    print(f"  {fn:<22s} {n:>7,} satir  OK")

import nltk
try: nltk.data.find("corpora/words")
except LookupError: nltk.download("words", quiet=True)

In [ ]:
# Hücre 2: IAM görüntü klasörü (words.txt'ye artık ihtiyaç yok)
import subprocess, os
res = subprocess.run(["find","/kaggle/input","-maxdepth","6","-type","d","-name","words"], capture_output=True, text=True)
IAM_ROOT = None
for d in [p.strip() for p in res.stdout.splitlines() if p.strip()]:
    if os.path.isdir(os.path.join(d,"a01")): IAM_ROOT = d; break
assert IAM_ROOT, "IAM words/ görüntü klasörü bulunamadı"
print("IAM words/ :", IAM_ROOT)
n_png = sum(len(f) for _,_,f in os.walk(IAM_ROOT)); print(f"png sayısı  : {n_png:,}  (115,320 bekleniyor)")
assert n_png >= 115_320, f"görüntü eksik: {n_png:,} < 115,320"

---
## Split doğrulaması

Eğitime başlamadan önce bölmenin bütünlüğü kontrol edilir: form/yazar/metin ayrıklığı, resmi listeyle eşleşme, kayıt ve görüntü bütünlüğü. Bir kontrol bile geçmezse hücre hata verir ve eğitim başlamaz.

In [ ]:
subprocess.run(["python","verify_aachen_splits.py","--img-root",IAM_ROOT], check=True)

---
## Eğitimler
Seçilen oturumun modları sırayla eğitilir. Sadece `--aug-mode` değişir; epoch/batch/lr/patience/seed sabit.

In [ ]:
MODES = {"A": ["narrow", "full"], "B": ["photo", "elastic", "morph"]}[SESSION]
NARROW_CSV = "/kaggle/working/abl_narrow/test_results_analysis.csv"

for mode in MODES:
    print("\n" + "="*70 + f"\n  --aug-mode {mode}\n" + "="*70, flush=True)
    t0 = time.time()
    cmd = ["python","cloud/v3_augmented_train.py","--aug-mode",mode,
           "--epochs","100","--batch","128","--lr","7e-4","--patience","15",
           "--model-dir",f"/kaggle/working/abl_{mode}",
           "--iam-root",IAM_ROOT]
    # McNemar: onerilen model (full) ayni oturumda egitilen baseline (narrow) ile
    # karsilastirilir. Script'in varsayilan arayisi "Model_abl_narrow" oldugu icin
    # yolu acikca veriyoruz, yoksa test atlaniyor.
    if mode == "full" and os.path.exists(NARROW_CSV):
        cmd += ["--baseline-csv", NARROW_CSV]
        print(f"  McNemar baseline: {NARROW_CSV}")
    subprocess.run(cmd, check=True)
    print(f"\n  [{mode}] süre: {(time.time()-t0)/3600:.2f} saat", flush=True)

---
## Tablo B — Lexicon / trigram ablation (yalnız Oturum A)
`full` modelinin greedy çıktısına dört post-processing uygulanır; eğitim yok.

In [ ]:
if SESSION == "A":
    subprocess.run(["python","cloud/ablation_lexicon.py",
                    "--model","/kaggle/working/abl_full/best_model_wa.pth",
                    "--iam-root",IAM_ROOT,
                    "--out","/kaggle/working/results/ablation_lexicon.json"], check=True)
else:
    print("Oturum B: lexicon ablation atlandı (Oturum A'da yapılır)")


---
## Özet tablo

In [ ]:
import json, os, csv, math

def wa_cer(path):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    k = sum(1 for r in rows
            if str(r.get("correct", r.get("Is_Correct", ""))).strip().lower() in ("1", "true"))
    n = len(rows); p = k / n; z = 1.96
    lo = (p + z*z/(2*n) - z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1 + z*z/n)
    hi = (p + z*z/(2*n) + z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1 + z*z/n)
    return k, n, p*100, lo*100, hi*100

LABEL = {"narrow":  "CRNN-L (baseline)",
         "photo":   "+ wide photometric",
         "elastic": "+ elastic",
         "morph":   "+ morphological",
         "full":    "AugCRNN-T (proposed)"}

print("=" * 78)
print(" TABLO A - Augmentation ablation  (tam Aachen test, N=20,310)")
print(" WA sutunu greedy + trigram sonrasidir (tum satirlarda ayni pipeline)")
print("=" * 78)
print(f"{'Configuration':<26s}{'WA (%)':>9s}{'95% CI':>18s}{'k/n':>16s}")
for mode in ["narrow", "photo", "elastic", "morph", "full"]:
    c = f"/kaggle/working/abl_{mode}/test_results_analysis.csv"
    if os.path.exists(c):
        k, n, wa, lo, hi = wa_cer(c)
        print(f"{LABEL[mode]:<26s}{wa:>9.2f}{f'[{lo:.2f}, {hi:.2f}]':>18s}{f'{k}/{n}':>16s}")
    else:
        print(f"{LABEL[mode]:<26s}{'-':>9s}{'(bu oturumda yok)':>18s}")

# McNemar (full vs narrow) - full egitimi sirasinda hesaplanip results.json'a yazilir
rj = "/kaggle/working/abl_full/results.json"
if os.path.exists(rj):
    mc = json.load(open(rj)).get("mcnemar_vs_v3_base") or {}
    if mc:
        print(f"\nMcNemar (full vs narrow): baseline {mc['baseline_wa_pct']:.2f}%  "
              f"delta {mc['delta_pp']:+.2f}pp  p = {mc['mcnemar_p']:.3e}")

p = "/kaggle/working/results/ablation_lexicon.json"
if os.path.exists(p):
    r = json.load(open(p))
    print("\n" + "=" * 78)
    print(" TABLO B - Lexicon / trigram ablation")
    print("=" * 78)
    print(f"{'Configuration':<42s}{'WA (%)':>9s}{'CER (%)':>9s}{'95% CI':>20s}")
    for c in r["configurations"]:
        ci = c["wilson_95ci_pct"]
        print(f"{c['name']:<42s}{c['wa_pct']:>9.2f}{c['cer_pct']:>9.2f}"
              f"{f'[{ci[0]:.2f}, {ci[1]:.2f}]':>20s}")
    print(f"\nLexicon boyutlari: {r['lexicon_sizes']}   N = {r['n_samples']:,}")
    print("Tutarlilik: Tablo B'nin son satiri, Tablo A'nin 'AugCRNN-T (proposed)'"
          " satiriyla ayni olmali.")

---
## Teslim paketi
Gönderilmesi gereken dosyalar tek klasöre toplanır ve `ablation_SESSION_<A|B>.zip` olarak paketlenir.
Output sekmesinden sadece bu zip'i indirmek yeterli.

In [ ]:
import shutil, os, glob
OUT = f"/kaggle/working/deliver_{SESSION}"
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT, exist_ok=True)

n = 0
for mode in ["narrow","photo","elastic","morph","full"]:
    d = f"/kaggle/working/abl_{mode}"
    if not os.path.isdir(d): continue
    dst = os.path.join(OUT, f"abl_{mode}"); os.makedirs(dst, exist_ok=True)
    for fn in ["test_results_analysis.csv","results.json","training_history.json",
               "test_summary_analysis.txt"]:
        p = os.path.join(d, fn)
        if os.path.exists(p): shutil.copy(p, dst); n += 1
for p in glob.glob("/kaggle/working/results/*.json"):
    shutil.copy(p, OUT); n += 1

zip_path = shutil.make_archive(f"/kaggle/working/ablation_SESSION_{SESSION}", "zip", OUT)
print(f"{n} dosya toplandı -> {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)\n")
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, OUT):<45s} {os.path.getsize(p)/1024:>9,.0f} KB")